# Spark Exercises 08 — Schemas, Reading & Writing

In pandas you `read_csv` and move on. In Spark — and especially in Foundry — **the schema is a first-class decision**. `inferSchema` is convenient but reads your data an extra time; in production you give Spark an explicit schema. This chapter covers schema-on-read, casting dirty string columns, and writing **Parquet** — including `partitionBy`, the layout that powers fast, partition-pruned reads on real datasets.

**Data files:** `data/orders.csv`, `data/customers.csv`

---

> **Solutions version** — try it yourself first from `Exercises.ipynb`.

### 1. **Setup** — open a local `SparkSession` (already written for you). In Foundry the session is created for you; locally we open one ourselves. `F` is the functions module — almost every column expression lives there.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-course")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 3.5.1


### 2. Read `data/orders.csv` with **just** `header=True` (no `inferSchema`). `printSchema()` — **everything is a string**, even `quantity` and `order_ts`. This is the default and it is fast.

In [2]:
raw = spark.read.csv("data/orders.csv", header=True)
raw.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- product_sku: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)
 |-- discount_code: string (nullable = true)



### 3. Now read it **with** `inferSchema=True` and `printSchema()`. Convenient — but Spark had to scan the file an **extra time** to guess the types.

In [3]:
inferred = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
inferred.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- product_sku: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)
 |-- discount_code: string (nullable = true)



### 4. **Best practice: an explicit schema.** Build a `StructType` for the orders file and read with `schema=...` (no inference pass, no surprises). `printSchema()` to confirm.

In [4]:
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType)

orders_schema = StructType([
    StructField("order_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("order_ts", StringType()),
    StructField("product_sku", StringType()),
    StructField("quantity", IntegerType()),
    StructField("unit_price", DoubleType()),
    StructField("channel", StringType()),
    StructField("payment_method", StringType()),
    StructField("status", StringType()),
    StructField("discount_code", StringType()),
])
orders = spark.read.csv("data/orders.csv", header=True, schema=orders_schema)
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- product_sku: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)
 |-- discount_code: string (nullable = true)



### 5. Starting from the all-string `raw`, **cast** `quantity` to int and `unit_price` to double, and parse `order_ts` into a real timestamp with `F.to_timestamp(..., "yyyy-MM-dd HH:mm:ss")`. Show the schema and 3 rows.

In [5]:
typed = (raw
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("order_ts", F.to_timestamp("order_ts", "yyyy-MM-dd HH:mm:ss")))
typed.printSchema()
typed.select("order_id", "quantity", "unit_price", "order_ts").show(3)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- product_sku: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)
 |-- discount_code: string (nullable = true)

+----------+--------+----------+-------------------+
|  order_id|quantity|unit_price|           order_ts|
+----------+--------+----------+-------------------+
|ORD-000001|      10|     32.58|2024-07-24 11:01:00|
|ORD-000002|      22|      1.43|2024-05-09 14:53:00|
|ORD-000003|       9|      1.14|2024-05-12 20:54:00|
+----------+--------+----------+-------------------+
only showing top 3 rows



### 6. Now that `order_ts` is a real timestamp, extract `year` and `month`, and count orders **per month**.

In [6]:
typed.withColumn("year", F.year("order_ts")) \
     .withColumn("month", F.month("order_ts")) \
     .groupBy("year", "month").count().orderBy("year", "month").show()

+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2024|    1|  642|
|2024|    2|  637|
|2024|    3|  672|
|2024|    4|  666|
|2024|    5|  702|
|2024|    6|  669|
|2024|    7|  668|
|2024|    8|  676|
|2024|    9|  643|
|2024|   10|  653|
|2024|   11|  717|
|2024|   12|  655|
+----+-----+-----+



### 7. **Write Parquet.** Create a temp output folder, then write `typed` as Parquet with `mode("overwrite")`. List the folder — note Parquet is a **directory** of part-files, not one file.

In [7]:
import tempfile, os
OUT = tempfile.mkdtemp()
path = os.path.join(OUT, "orders_parquet")
typed.write.mode("overwrite").parquet(path)
print(sorted(os.listdir(path))[:5])

['._SUCCESS.crc', '.part-00000-bc649da3-cc61-412a-ae35-f0071414461d-c000.snappy.parquet.crc', '_SUCCESS', 'part-00000-bc649da3-cc61-412a-ae35-f0071414461d-c000.snappy.parquet']


### 8. **Read the Parquet back.** Parquet stores the schema inside the files — so no `inferSchema` is needed and types come back exactly. `printSchema()` to prove it.

In [8]:
back = spark.read.parquet(path)
back.printSchema()
back.count()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- product_sku: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)
 |-- discount_code: string (nullable = true)



8000

### 9. **Partitioned write.** Write `typed` as Parquet **partitioned by `channel`** into a new temp path. List the directory — you'll see `channel=web`, `channel=app`, … sub-folders.

In [9]:
ppath = os.path.join(OUT, "orders_by_channel")
typed.write.mode("overwrite").partitionBy("channel").parquet(ppath)
print(sorted(os.listdir(ppath)))

['._SUCCESS.crc', '_SUCCESS', 'channel=app', 'channel=partner', 'channel=store', 'channel=web']


### 10. **Partition pruning.** Read the partitioned dataset and filter `channel == 'web'`, then `.explain()`. Look for `PartitionFilters` — Spark reads **only the `channel=web` folder**, skipping the rest.

In [10]:
part = spark.read.parquet(ppath)
part.filter(F.col("channel") == "web").explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [order_id#287,customer_id#288,order_ts#289,product_sku#290,quantity#291,unit_price#292,payment_method#293,status#294,discount_code#295,channel#296] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/tmp/tmpdglfkgwl/orders_by_channel], PartitionFilters: [isnotnull(channel#296), (channel#296 = web)], PushedFilters: [], ReadSchema: struct<order_id:string,customer_id:string,order_ts:timestamp,product_sku:string,quantity:int,unit...




### 11. Finally, write a single CSV (coalesce to 1 file) of the per-channel order counts, with a header, then read it back to confirm.

In [11]:
summary = typed.groupBy("channel").count()
cpath = os.path.join(OUT, "channel_summary_csv")
summary.coalesce(1).write.mode("overwrite").option("header", True).csv(cpath)
spark.read.csv(cpath, header=True, inferSchema=True).show()

+-------+-----+
|channel|count|
+-------+-----+
|    web| 3573|
|    app| 2400|
|partner|  565|
|  store| 1462|
+-------+-----+

